# Censo Argentino 2022 — Guía de uso con `ciut`

Este notebook muestra cómo consultar el **Censo Nacional de Población, Hogares y Viviendas 2022** (INDEC) con la librería `ciut-censo`.

## ¿Qué son estos datos?

El censo 2022 relevó **~47 millones de personas** en **~520.000 radios censales** de todo Argentina.

Los datos están organizados en tres entidades:

| Entidad | Qué mide | Ejemplos de variables |
|---|---|---|
| **PERSONA** | Características de cada persona | Sexo (`PERSONA_P02`), edad (`PERSONA_EDAD`), educación (`PERSONA_MNI`) |
| **HOGAR** | Características del grupo familiar | NBI (`HOGAR_NBI_TOT`), hacinamiento (`HOGAR_H20CP`) |
| **VIVIENDA** | Características del lugar | Tipo (`VIVIENDA_TIPOVIVG`), servicios (`VIVIENDA_URP`) |

## Estructura del resultado

Los datos vienen en **formato largo pre-agregado**: cada fila es una combinación de *(radio censal × categoría × conteo)*.

```
id_geo     | codigo_variable | valor_categoria | etiqueta_categoria   | conteo
460070101  | PERSONA_P02     | 1               | Mujer / Femenino     | 252
460070101  | PERSONA_P02     | 2               | Varón / Masculino    | 231
```

**Importante sobre las columnas geográficas:**
- `etiqueta_provincia` → nombre real (ej. "Córdoba")
- `etiqueta_departamento` → código numérico (ej. "007"), **no el nombre**
- Para obtener nombres de departamento: cruzar con la variable `DPTO_NDPTO`

## Instalación
```bash
pip install -e ciut-censo/           # básico
pip install -e "ciut-censo/[geo]"    # + geopandas para mapas
```

In [18]:
pip install ciut-censo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import ciut
import pandas as pd
print("OK")

OK


---
## 1. Explorar variables disponibles

In [22]:
# Todas las variables
ciut.variables()

,codigo_variable,etiqueta_variable,entidad
0,DPTO_IDPTO,Departamento,DPTO
1,DPTO_NDPTO,Departamento,DPTO
2,DPTO_REDCODEN,Departamento,DPTO
3,FRAC_IDFRAC,Fraccion,FRAC
4,FRAC_REDCODEN,Fracción,FRAC
...,...,...,...
81,VIVIENDA_V01,Tipo de vivienda,VIVIENDA
82,VIVIENDA_V01_1,Tipo de vivienda particular ocupada,VIVIENDA
83,VIVIENDA_V02,Condición de ocupación,VIVIENDA
84,VIVIENDA_V04,Motivo por el que no realizó la entrevista,VIVIENDA


In [3]:
# Solo variables de personas
ciut.variables(entidad="PERSONA")

,codigo_variable,etiqueta_variable,entidad
0,PERSONA_AESC,Años de escolaridad completos,PERSONA
1,PERSONA_CONASI,Condición de asistencia escolar,PERSONA
2,PERSONA_CONDACT,Condición de actividad económica,PERSONA
3,PERSONA_CONDACTD,Condición de actividad económica combinada,PERSONA
4,PERSONA_DESCAPOR,Descuento o aporte jubilatorio,PERSONA
5,PERSONA_EDAD,Edad,PERSONA
6,PERSONA_EDADGRU,Edad en grandes grupos,PERSONA
7,PERSONA_EDADQUI,Edad en grupos quinquenales,PERSONA
8,PERSONA_LETRA,Rama de actividad económica (primer nivel),PERSONA
9,PERSONA_MNI,Máximo nivel de instrucción alcanzado,PERSONA


In [23]:
# Buscar por tema
ciut.variables(buscar="edad")

,codigo_variable,etiqueta_variable,entidad
0,HOGAR_H22,Propiedad de la vivienda,HOGAR
1,PERSONA_EDAD,Edad,PERSONA
2,PERSONA_EDADGRU,Edad en grandes grupos,PERSONA
3,PERSONA_EDADQUI,Edad en grupos quinquenales,PERSONA


In [5]:
ciut.variables(buscar="nbi")

,codigo_variable,etiqueta_variable,entidad
0,HOGAR_NBI_ESC,NBI Escolaridad,HOGAR
1,HOGAR_NBI_HAC,NBI Hacinamiento,HOGAR
2,HOGAR_NBI_SAN,NBI Condiciones sanitarias,HOGAR
3,HOGAR_NBI_SUB,NBI Capacidad de subsistencia,HOGAR
4,HOGAR_NBI_TOT,Necesidades básicas insatisfechas,HOGAR
5,HOGAR_NBI_VIV,NBI Vivienda de tipo inconveniente,HOGAR


In [6]:
ciut.variables(buscar="instruccion")

,codigo_variable,etiqueta_variable,entidad


---
## 2. Entender una variable: `describe()`

Muestra la descripción y todas las categorías posibles.

In [7]:
# Sexo registrado al nacer (la variable se llama P02, no SEXO)
ciut.describe("PERSONA_P02")

[ciut] Consultando metadatos de 'PERSONA_P02'...

  Variable   : PERSONA_P02
  Nombre INDEC: P02
  Descripción: Sexo registrado al nacer
  Entidad    : PERSONA  (aplica a personas)
  Referencia : https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf

  Categorías (2 valores):
  Código      Etiqueta
  --------  ----------------------------------------
  1           Mujer / Femenino
  2           Varón / Masculino



In [8]:
# Edad en grupos quinquenales (00-04, 05-09, ..., 105 y más)
ciut.describe("PERSONA_EDADQUI")

[ciut] Consultando metadatos de 'PERSONA_EDADQUI'...

  Variable   : PERSONA_EDADQUI
  Nombre INDEC: EDADQUI
  Descripción: Edad en grupos quinquenales
  Entidad    : PERSONA  (aplica a personas)
  Referencia : https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf

  Categorías (22 valores):
  Código      Etiqueta
  --------  ----------------------------------------
  1           00 A 04
  2           05 A 09
  3           10 A 14
  4           15 A 19
  5           20 A 24
  6           25 A 29
  7           30 A 34
  8           35 A 39
  9           40 A 44
  10          45 A 49
  11          50 A 54
  12          55 A 59
  13          60 A 64
  14          65 A 69
  15          70 A 74
  16          75 A 79
  17          80 A 84
  18          85 A 89
  19          90 A 94
  20          95 A 99
  21          100 A 104
  22          105 Y MÁS



In [9]:
# Edad en grandes grupos (0-14, 15-64, 65+)
ciut.describe("PERSONA_EDADGRU")

[ciut] Consultando metadatos de 'PERSONA_EDADGRU'...

  Variable   : PERSONA_EDADGRU
  Nombre INDEC: EDADGRU
  Descripción: Edad en grandes grupos
  Entidad    : PERSONA  (aplica a personas)
  Referencia : https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf

  Categorías (3 valores):
  Código      Etiqueta
  --------  ----------------------------------------
  1           HASTA 14 AÑOS
  2           15 A 64 AÑOS
  3           65 AÑOS Y MÁS



In [10]:
# Máximo nivel de instrucción
ciut.describe("PERSONA_MNI")

[ciut] Consultando metadatos de 'PERSONA_MNI'...

  Variable   : PERSONA_MNI
  Nombre INDEC: MNI
  Descripción: Máximo nivel de instrucción alcanzado
  Entidad    : PERSONA  (aplica a personas)
  Referencia : https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf

  Categorías (12 valores):
  Código      Etiqueta
  --------  ----------------------------------------
  1           Sin instrucción
  2           Primario incompleto
  3           Primario completo
  4           Secundario incompleto
  5           Secundario completo
  6           Terciario incompleto
  7           Terciario completo
  8           Universitario incompleto
  9           Universitario completo
  10          Posgrado incompleto
  11          Posgrado completo
  99          Ignorado



In [11]:
# Condición de actividad económica
ciut.describe("PERSONA_CONDACT")

[ciut] Consultando metadatos de 'PERSONA_CONDACT'...

  Variable   : PERSONA_CONDACT
  Nombre INDEC: CONDACT
  Descripción: Condición de actividad económica
  Entidad    : PERSONA  (aplica a personas)
  Referencia : https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf

  Categorías (3 valores):
  Código      Etiqueta
  --------  ----------------------------------------
  1           Ocupado
  2           Desocupado
  3           Inactivo



In [12]:
# NBI total del hogar
ciut.describe("HOGAR_NBI_TOT")

[ciut] Consultando metadatos de 'HOGAR_NBI_TOT'...

  Variable   : HOGAR_NBI_TOT
  Nombre INDEC: NBI_TOT
  Descripción: Necesidades básicas insatisfechas
  Entidad    : HOGAR  (aplica a hogares)
  Referencia : https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf

  Categorías (2 valores):
  Código      Etiqueta
  --------  ----------------------------------------
  1           Sí
  2           No



In [13]:
# Tipo de vivienda agrupado
ciut.describe("VIVIENDA_TIPOVIVG")

[ciut] Consultando metadatos de 'VIVIENDA_TIPOVIVG'...

  Variable   : VIVIENDA_TIPOVIVG
  Nombre INDEC: TIPOVIVG
  Descripción: Tipo de vivienda agrupado
  Entidad    : VIVIENDA  (aplica a viviendas)
  Referencia : https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf

  Categorías (1 valores):
  Código      Etiqueta
  --------  ----------------------------------------
  1           Particular



---
## 3. Provincias disponibles

In [14]:
# Acepta nombre o código INDEC como filtro en query()
ciut.provincias()

,codigo,provincia
0,02,Ciudad Autónoma De Buenos Aires
1,06,Buenos Aires
2,10,Catamarca
3,14,Córdoba
4,18,Corrientes
5,22,Chaco
6,26,Chubut
7,30,Entre Ríos
8,34,Formosa
9,38,Jujuy


---
## 4. Conteos básicos por provincia

Usamos **La Rioja (código 46)** en los primeros ejemplos porque es la provincia más chica y descarga rápido.

In [15]:
# Sexo en La Rioja
df_sexo = ciut.query(variables="PERSONA_P02", provincia="46")
df_sexo.head(6)

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : PERSONA_P02  ("Sexo registrado al nacer")
[ciut]   Provincia: La Rioja  (código INDEC: 46)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...


RuntimeError: Query interrupted

In [ ]:
# Total de personas por sexo en La Rioja
# Nota: código 1 = Mujer/Femenino, código 2 = Varón/Masculino
(
    df_sexo
    .groupby("etiqueta_categoria")["conteo"]
    .sum()
    .rename("total_personas")
    .reset_index()
)

,etiqueta_categoria,total_personas
0,Mujer / Femenino,196187
1,Varón / Masculino,186266


In [ ]:
# Porcentaje por sexo
total = df_sexo["conteo"].sum()
(
    df_sexo
    .groupby("etiqueta_categoria")["conteo"]
    .sum()
    .rename("personas")
    .to_frame()
    .assign(pct=lambda d: (d["personas"] / total * 100).round(1))
)

,personas,pct
etiqueta_categoria,,
Mujer / Femenino,196187,51.3
Varón / Masculino,186266,48.7


---
## 5. Grupos de edad por provincia

Hay tres variables de edad:
- `PERSONA_EDADGRU` → 3 grupos grandes (0-14, 15-64, 65+)
- `PERSONA_EDADQUI` → 22 grupos quinquenales (00-04, 05-09, ...)
- `PERSONA_EDAD` → edad exacta en años (0 a 110+)

In [ ]:
# Grandes grupos de edad en La Rioja
df_edadgru = ciut.query(variables="PERSONA_EDADGRU", provincia="46")

(
    df_edadgru
    .groupby("etiqueta_categoria")["conteo"]
    .sum()
    .rename("personas")
    .reset_index()
)

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : PERSONA_EDADGRU  ("Edad en grandes grupos")
[ciut]   Provincia: La Rioja  (código INDEC: 46)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 4.7s -> 1,491 filas | 13 columnas | 1.0 MB en memoria


,etiqueta_categoria,personas
0,15 A 64 AÑOS,263098
1,65 AÑOS Y MÁS,34638
2,HASTA 14 AÑOS,84717


In [ ]:
# Grupos quinquenales en La Rioja
# valor_categoria es el orden (1=00-04, 2=05-09, ...) → ordenamos por él
df_edadqui = ciut.query(variables="PERSONA_EDADQUI", provincia="46")

piramide = (
    df_edadqui
    .groupby(["valor_categoria", "etiqueta_categoria"])["conteo"]
    .sum()
    .reset_index()
    .sort_values("valor_categoria")
    .rename(columns={"etiqueta_categoria": "grupo_edad", "conteo": "personas"})
    .drop(columns="valor_categoria")
    .reset_index(drop=True)
)
piramide

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : PERSONA_EDADQUI  ("Edad en grupos quinquenales")
[ciut]   Provincia: La Rioja  (código INDEC: 46)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 5.0s -> 9,404 filas | 13 columnas | 5.8 MB en memoria


,grupo_edad,personas
0,00 A 04,23655
1,45 A 49,23620
2,50 A 54,19418
3,55 A 59,17175
4,60 A 64,15075
5,65 A 69,12467
6,70 A 74,9305
7,75 A 79,6248
8,80 A 84,3641
9,85 A 89,1845


In [ ]:
# Comparar grupos de edad entre varias provincias
# Usamos PERSONA_EDADGRU (3 grupos) para que sea manejable
provincias_comparar = ["46", "02", "06", "14"]  # La Rioja, CABA, Bs As, Córdoba

frames = []
for prov in provincias_comparar:
    df = ciut.query(variables="PERSONA_EDADGRU", provincia=prov)
    frames.append(df)

df_multi = pd.concat(frames, ignore_index=True)

# Tabla: provincia × grupo de edad
tabla_edad = (
    df_multi
    .groupby(["etiqueta_provincia", "etiqueta_categoria"])["conteo"]
    .sum()
    .unstack("etiqueta_categoria")
    .fillna(0)
    .astype(int)
)
tabla_edad

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : PERSONA_EDADGRU  ("Edad en grandes grupos")
[ciut]   Provincia: La Rioja  (código INDEC: 46)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 0.7s -> 1,491 filas | 13 columnas | 1.0 MB en memoria
[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (IND

etiqueta_categoria,15 A 64 AÑOS,65 AÑOS Y MÁS,HASTA 14 AÑOS
etiqueta_provincia,,,
Buenos Aires,11467152,2113084,3828670
Caba,2098574,537004,459876
Córdoba,2531392,474062,806610
La Rioja,263098,34638,84717


In [ ]:
# Porcentaje por grupo de edad (por provincia)
tabla_pct = tabla_edad.div(tabla_edad.sum(axis=1), axis=0).mul(100).round(1)
tabla_pct

etiqueta_categoria,15 A 64 AÑOS,65 AÑOS Y MÁS,HASTA 14 AÑOS
etiqueta_provincia,,,
Buenos Aires,65.9,12.1,22.0
Caba,67.8,17.3,14.9
Córdoba,66.4,12.4,21.2
La Rioja,68.8,9.1,22.2


---
## 6. Nivel de instrucción

Variable `PERSONA_MNI` — Máximo nivel de instrucción alcanzado.

In [ ]:
df_educ = ciut.query(variables="PERSONA_MNI", provincia="14")  # Córdoba

resumen_educ = (
    df_educ
    .groupby(["valor_categoria", "etiqueta_categoria"])["conteo"]
    .sum()
    .reset_index()
    .sort_values("valor_categoria")
    .rename(columns={"etiqueta_categoria": "nivel", "conteo": "personas"})
    .drop(columns="valor_categoria")
    .reset_index(drop=True)
)
# Excluir ignorados (código 99) del total
resumen_educ

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : PERSONA_MNI  ("Máximo nivel de instrucción alcanzado")
[ciut]   Provincia: Córdoba  (código INDEC: 14)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 5.5s -> 76,386 filas | 13 columnas | 49.0 MB en memoria


,nivel,personas
0,Sin instrucción,351381
1,Posgrado incompleto,32521
2,Posgrado completo,54230
3,Primario incompleto,526659
4,Primario completo,417316
5,Secundario incompleto,861021
6,Secundario completo,590413
7,Terciario incompleto,155298
8,Terciario completo,222762
9,Universitario incompleto,350145


In [ ]:
# Porcentaje (excluyendo código 99 = ignorado)
sin_ignorados = resumen_educ[resumen_educ["nivel"] != "Ignorado"].copy()
total_educ = sin_ignorados["personas"].sum()
sin_ignorados["pct"] = (sin_ignorados["personas"] / total_educ * 100).round(1)
sin_ignorados

,nivel,personas,pct
0,Sin instrucción,351381,9.3
1,Posgrado incompleto,32521,0.9
2,Posgrado completo,54230,1.4
3,Primario incompleto,526659,13.9
4,Primario completo,417316,11.0
5,Secundario incompleto,861021,22.7
6,Secundario completo,590413,15.6
7,Terciario incompleto,155298,4.1
8,Terciario completo,222762,5.9
9,Universitario incompleto,350145,9.2


---
## 7. NBI — Necesidades Básicas Insatisfechas

El INDEC calcula 5 componentes de NBI + un total.

In [ ]:
# Las 6 variables de NBI disponibles
ciut.variables(buscar="nbi")

,codigo_variable,etiqueta_variable,entidad
0,HOGAR_NBI_ESC,NBI Escolaridad,HOGAR
1,HOGAR_NBI_HAC,NBI Hacinamiento,HOGAR
2,HOGAR_NBI_SAN,NBI Condiciones sanitarias,HOGAR
3,HOGAR_NBI_SUB,NBI Capacidad de subsistencia,HOGAR
4,HOGAR_NBI_TOT,Necesidades básicas insatisfechas,HOGAR
5,HOGAR_NBI_VIV,NBI Vivienda de tipo inconveniente,HOGAR


In [ ]:
# NBI total en Tucumán
df_nbi = ciut.query(variables="HOGAR_NBI_TOT", provincia="Tucumán")
ciut.describe("HOGAR_NBI_TOT")

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : HOGAR_NBI_TOT  ("Necesidades básicas insatisfechas")
[ciut]   Provincia: Tucumán  (código INDEC: 90)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 5.3s -> 4,560 filas | 13 columnas | 2.9 MB en memoria
[ciut] Consultando metadatos de 'HOGAR_NBI_TOT'...

  Variable   : HOGAR_NBI_TOT
  Nombre INDEC: N

In [ ]:
# Hogares con y sin NBI en Tucumán
resumen_nbi = (
    df_nbi
    .groupby("etiqueta_categoria")["conteo"]
    .sum()
    .rename("hogares")
    .reset_index()
)
total_h = resumen_nbi["hogares"].sum()
resumen_nbi["pct"] = (resumen_nbi["hogares"] / total_h * 100).round(1)
resumen_nbi

,etiqueta_categoria,hogares,pct
0,No,457470,90.5
1,Sí,48072,9.5


In [ ]:
# Comparar % de hogares con NBI entre todas las provincias
# (trae datos de todo el país para esta variable)
df_nbi_pais = ciut.query(variables="HOGAR_NBI_TOT")

# Filtrar solo los que tienen NBI (la etiqueta exacta depende del describe())
# Exploramos qué valores hay
df_nbi_pais.groupby("etiqueta_categoria")["conteo"].sum().reset_index()

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : HOGAR_NBI_TOT  ("Necesidades básicas insatisfechas")
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 0.8s -> 127,347 filas | 13 columnas | 79.3 MB en memoria


,etiqueta_categoria,conteo
0,No,14858011
1,Sí,1074291


In [ ]:
# % hogares con NBI por provincia (ordenado de mayor a menor)
por_provincia = (
    df_nbi_pais
    .groupby(["etiqueta_provincia", "etiqueta_categoria"])["conteo"]
    .sum()
    .unstack("etiqueta_categoria")
    .fillna(0)
)

# La columna con NBI es la que no es "Sin NBI" — usamos las columnas reales del resultado
print("Columnas disponibles:", por_provincia.columns.tolist())

Columnas disponibles: ['No', 'Sí']


In [ ]:
# Completar luego de ver el output de arriba:
# Ejemplo asumiendo que las categorías son "Con NBI" y "Sin NBI"
# (verificar con ciut.describe("HOGAR_NBI_TOT"))
total_por_prov = por_provincia.sum(axis=1)
pct_nbi_prov = (por_provincia.div(total_por_prov, axis=0) * 100).round(1)
pct_nbi_prov.sort_values(pct_nbi_prov.columns[0], ascending=False)

etiqueta_categoria,No,Sí
etiqueta_provincia,,
La Pampa,97.4,2.6
Córdoba,95.7,4.3
Caba,95.2,4.8
Santa Fe,95.1,4.9
Chubut,94.9,5.1
Río Negro,94.1,5.9
Santa Cruz,94.0,6.0
Entre Ríos,94.0,6.0
Buenos Aires,93.7,6.3


---
## 8. Condición de actividad económica

In [ ]:
# Ocupados / Desocupados / Inactivos en Santa Fe
df_act = ciut.query(variables="PERSONA_CONDACT", provincia="Santa Fe")

resumen_act = (
    df_act
    .groupby(["valor_categoria", "etiqueta_categoria"])["conteo"]
    .sum()
    .reset_index()
    .sort_values("valor_categoria")
    .rename(columns={"etiqueta_categoria": "condicion", "conteo": "personas"})
    .drop(columns="valor_categoria")
    .reset_index(drop=True)
)
total_pea = resumen_act.loc[resumen_act["condicion"].isin(["Ocupado", "Desocupado"]), "personas"].sum()
resumen_act["pct_total"] = (resumen_act["personas"] / resumen_act["personas"].sum() * 100).round(1)
resumen_act

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : PERSONA_CONDACT  ("Condición de actividad económica")
[ciut]   Provincia: Santa Fe  (código INDEC: 82)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 5.6s -> 15,950 filas | 13 columnas | 9.9 MB en memoria


,condicion,personas,pct_total
0,Ocupado,1667277,59.0
1,Desocupado,144646,5.1
2,Inactivo,1012292,35.8


In [ ]:
# Tasa de desocupación (desocupados / PEA)
ocupados = resumen_act.loc[resumen_act["condicion"] == "Ocupado", "personas"].values[0]
desocupados = resumen_act.loc[resumen_act["condicion"] == "Desocupado", "personas"].values[0]
pea = ocupados + desocupados

print(f"PEA (ocupados + desocupados): {pea:,}")
print(f"Tasa de desocupacion: {desocupados / pea * 100:.1f}%")

PEA (ocupados + desocupados): 1,811,923
Tasa de desocupacion: 8.0%


---
## 9. Tipo de vivienda

In [ ]:
# Tipo de vivienda en CABA
df_viv = ciut.query(variables="VIVIENDA_TIPOVIVG", provincia="02")

dist_viv = (
    df_viv
    .groupby("etiqueta_categoria")["conteo"]
    .sum()
    .sort_values(ascending=False)
    .rename("viviendas")
    .reset_index()
)
dist_viv["pct"] = (dist_viv["viviendas"] / dist_viv["viviendas"].sum() * 100).round(1)
dist_viv

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : VIVIENDA_TIPOVIVG  ("Tipo de vivienda agrupado")
[ciut]   Provincia: Ciudad Autónoma De Buenos Aires  (código INDEC: 02)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 5.2s -> 3,820 filas | 13 columnas | 2.4 MB en memoria


,etiqueta_categoria,viviendas,pct
0,Particular,1614354,100.0


---
## 10. Acceso a internet y tecnología

In [ ]:
# Las tres variables de conectividad
ciut.describe("HOGAR_H24A")  # Internet en la vivienda

[ciut] Consultando metadatos de 'HOGAR_H24A'...

  Variable   : HOGAR_H24A
  Nombre INDEC: H24A
  Descripción: El hogar tiene internet en la vivienda
  Entidad    : HOGAR  (aplica a hogares)
  Referencia : https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf

  Categorías (2 valores):
  Código      Etiqueta
  --------  ----------------------------------------
  1           Si
  2           No



In [ ]:
# Internet, celular y computadora en Salta
df_tech = ciut.query(
    variables=["HOGAR_H24A", "HOGAR_H24B", "HOGAR_H24C"],
    provincia="Salta"
)

labels_var = {
    "HOGAR_H24A": "Internet en vivienda",
    "HOGAR_H24B": "Celular con internet",
    "HOGAR_H24C": "Computadora / tablet",
}

resumen_tech = (
    df_tech
    .groupby(["codigo_variable", "etiqueta_categoria"])["conteo"]
    .sum()
    .unstack("etiqueta_categoria")
    .fillna(0)
    .astype(int)
)
resumen_tech.index = resumen_tech.index.map(labels_var)
resumen_tech

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : HOGAR_H24A  ("El hogar tiene internet en la vivienda")
[ciut]   Variable : HOGAR_H24B  ("El hogar tiene celular con internet")
[ciut]   Variable : HOGAR_H24C  ("El hogar tiene computadora, tablet")
[ciut]   Provincia: Salta  (código INDEC: 66)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 6.0s -> 1

etiqueta_categoria,No,Si,Sí
codigo_variable,,,
Internet en vivienda,156227,262203,0
Celular con internet,74844,0,343586
Computadora / tablet,222213,0,196217


---
## 11. Nombres de departamentos

`etiqueta_departamento` solo tiene el código numérico (ej. "007").
Para obtener nombres reales cruzamos con la variable `DPTO_NDPTO`.

In [ ]:
# Nombres de departamentos de Córdoba
df_dpto = ciut.query(variables="DPTO_NDPTO", provincia="14")

# Lookup: código departamento -> nombre
dpto_nombres = (
    df_dpto[["valor_departamento", "etiqueta_categoria"]]
    .drop_duplicates()
    .rename(columns={"etiqueta_categoria": "nombre_departamento"})
    .sort_values("valor_departamento")
    .reset_index(drop=True)
)
dpto_nombres.head(15)

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : DPTO_NDPTO  ("Departamento")
[ciut]   Provincia: Córdoba  (código INDEC: 14)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 4.8s -> 6,720 filas | 13 columnas | 4.4 MB en memoria


,valor_departamento,nombre_departamento
0,007,Calamuchita
1,014,Capital
2,021,Colón
3,028,Cruz del Eje
4,035,General Roca
5,042,General San Martín
6,049,Ischilín
7,056,Juárez Celman
8,063,Marcos Juárez
9,070,Minas


In [ ]:
# Ahora combinamos nombres con datos de otra variable
df_educ_cba = ciut.query(variables="PERSONA_MNI", provincia="14")

# Unir nombres de departamento
df_educ_cba = df_educ_cba.merge(dpto_nombres, on="valor_departamento", how="left")

# Universitario completo por departamento
univ = (
    df_educ_cba[df_educ_cba["etiqueta_categoria"] == "Universitario completo"]
    .groupby("nombre_departamento")["conteo"]
    .sum()
    .sort_values(ascending=False)
    .rename("universitario_completo")
    .reset_index()
)
univ.head(15)

[ciut] =======================================================
[ciut] Consulta al Censo Nacional 2022 (INDEC)
[ciut] Fuente: censo-2022-largo.parquet (~2.1 GB total en S3)
[ciut]   (DuckDB descarga solo los bloques que coinciden con los filtros)
[ciut] -------------------------------------------------------
[ciut]   Variable : PERSONA_MNI  ("Máximo nivel de instrucción alcanzado")
[ciut]   Provincia: Córdoba  (código INDEC: 14)
[ciut] -------------------------------------------------------
[ciut] Estructura del resultado:
[ciut]   Cada fila = una (radio censal × categoría de variable × conteo)
[ciut]   Columnas clave: id_geo | codigo_variable | valor_categoria
[ciut]                   etiqueta_categoria | conteo
[ciut] =======================================================
[ciut] Descargando datos desde S3...
[ciut] Descarga completa en 0.9s -> 76,386 filas | 13 columnas | 49.0 MB en memoria


,nombre_departamento,universitario_completo
0,Capital,112809
1,Colón,18887
2,Río Cuarto,18560
3,Punilla,12080
4,San Justo,10850
5,General San Martín,7953
6,Santa Marķa,7805
7,Tercero Arriba,6153
8,Marcos Juárez,5043
9,Unión,4779


---
## 12. Mapas con geometría de radios censales

Requiere: `pip install -e "ciut-censo/[geo]"`

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd

# Grupos de edad en Mendoza con geometría
gdf = ciut.query(variables="PERSONA_EDADGRU", provincia="Mendoza", geometry=True)
print(gdf.shape)
gdf.head(4)

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Mapa: % de adultos mayores (65+) por radio censal en Mendoza

# Pivotar: radio x grupo de edad
pivot = (
    gdf
    .groupby(["id_geo", "etiqueta_categoria"])
    .agg(conteo=("conteo", "sum"), geometry=("geometry", "first"))
    .reset_index()
    .pivot_table(index=["id_geo", "geometry"], columns="etiqueta_categoria",
                 values="conteo", aggfunc="sum")
    .reset_index()
)

# Columnas tras el pivot — usamos las etiquetas reales del censo
col_65 = "65 ANOS Y MAS"      # verificar con ciut.describe("PERSONA_EDADGRU")
col_0  = "HASTA 14 ANOS"
col_15 = "15 A 64 ANOS"

pivot["total"] = pivot[col_0] + pivot[col_15] + pivot[col_65]
pivot["pct_65"] = pivot[col_65] / pivot["total"] * 100

gdf_map = gpd.GeoDataFrame(pivot, geometry="geometry", crs="EPSG:4326")

fig, ax = plt.subplots(figsize=(8, 10))
gdf_map.plot(column="pct_65", cmap="YlOrRd", legend=True, ax=ax,
             legend_kwds={"label": "% adultos mayores (65+)"},
             edgecolor="none", missing_kwds={"color": "lightgrey"})
ax.set_title("Adultos mayores (65+) por radio censal\nMendoza — Censo 2022")
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Mapa: % hogares con NBI en Tucumán
gdf_nbi = ciut.query(variables="HOGAR_NBI_TOT", provincia="Tucumán", geometry=True)

# Ver columnas exactas
print(gdf_nbi["etiqueta_categoria"].unique())

In [ ]:
# Pivotar y calcular % NBI (ajustar nombre de columna según output de arriba)
pivot_nbi = (
    gdf_nbi
    .groupby(["id_geo", "etiqueta_categoria"])
    .agg(conteo=("conteo", "sum"), geometry=("geometry", "first"))
    .reset_index()
    .pivot_table(index=["id_geo", "geometry"], columns="etiqueta_categoria",
                 values="conteo", aggfunc="sum")
    .reset_index()
)

print("Columnas:", pivot_nbi.columns.tolist())

---
## 13. Exportar resultados

In [ ]:
df = ciut.query(variables="PERSONA_EDADQUI", provincia="Salta")

# CSV
df.to_csv("salta_edad_quinquenal.csv", index=False)

# Excel
df.to_excel("salta_edad_quinquenal.xlsx", index=False)

# Parquet local (más eficiente para reusar sin volver a descargar)
df.to_parquet("salta_edad_quinquenal.parquet", index=False)

print("Archivos guardados")

---
## Referencia rápida

### Variables más usadas

| Variable | Descripción | Categorías |
|---|---|---|
| `PERSONA_P02` | Sexo registrado al nacer | 1=Mujer/Femenino, 2=Varón/Masculino |
| `PERSONA_EDADGRU` | Edad en grandes grupos | 1=0-14, 2=15-64, 3=65+ |
| `PERSONA_EDADQUI` | Edad quinquenal | 1=00-04, 2=05-09, ..., 22=105+ |
| `PERSONA_EDAD` | Edad exacta en años | 0 a 110+ |
| `PERSONA_MNI` | Máximo nivel de instrucción | 1=Sin instrucción ... 11=Posgrado completo |
| `PERSONA_CONDACT` | Condición de actividad | 1=Ocupado, 2=Desocupado, 3=Inactivo |
| `HOGAR_NBI_TOT` | NBI total del hogar | Con NBI / Sin NBI |
| `HOGAR_H20CP` | Hacinamiento | Sin hacinamiento / Con hacinamiento / ... |
| `HOGAR_H24A` | Internet en vivienda | Sí / No |
| `VIVIENDA_TIPOVIVG` | Tipo de vivienda | Particular, Colectiva, etc. |
| `VIVIENDA_URP` | Área urbano-rural | Urbano / Rural |
| `DPTO_NDPTO` | Nombre del departamento | (lookup de códigos a nombres) |

### Funciones

| Función | Qué hace |
|---|---|
| `ciut.variables()` | Lista todas las variables |
| `ciut.variables(entidad="PERSONA")` | Filtra por entidad |
| `ciut.variables(buscar="edad")` | Busca por texto |
| `ciut.describe("PERSONA_P02")` | Muestra categorías de una variable |
| `ciut.provincias()` | Tabla de nombres y códigos INDEC |
| `ciut.query(variables="X", provincia="Y")` | Descarga datos filtrados |
| `ciut.query(..., geometry=True)` | Agrega polígonos de radios censales |

### Documentación INDEC
- [Definiciones de variables (PDF)](https://redatam.indec.gob.ar/redarg/CENSOS/CPV2022/Docs/Redatam_Definiciones_de_la_base_de_datos.pdf)
- [Cuestionario del censo (PDF)](https://www.indec.gob.ar/ftp/cuadros/poblacion/Censo2022_cuestionario_viviendas_particulares.pdf)
- [Portal REDATAM online](https://redatam.indec.gob.ar/binarg/RpWebEngine.exe/Portal?BASE=CPV2022&lang=ESP)